# 🤖 智能数据更新系统

## 📚 功能介绍

这个Notebook提供了智能数据更新功能，可以：

1. **自动发现最新数据集** - 搜索GEE上所有数据类型的最新版本
2. **一键更新配置** - 自动更新所有数据源的配置
3. **智能推荐** - 基于版本、完整性和可访问性推荐最佳数据集
4. **安全更新** - 自动备份，支持回滚
5. **详细报告** - 生成完整的更新报告

---

## 🚀 快速开始（3步）

### 第1步：初始化环境

In [ ]:
# ============================================================
# 初始化环境
# ============================================================

import sys
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# 添加项目路径
project_root = Path.cwd()
sys.path.insert(0, str(project_root / 'core'))

print("📁 项目路径:", project_root)
print("\n📦 导入模块...")

# 导入核心库
import pandas as pd
import numpy as np
import json
from datetime import datetime

# 尝试导入GEE
try:
    import ee
    print("✅ Google Earth Engine 导入成功")
    
    # 初始化GEE
    try:
        ee.Initialize()
        print("✅ GEE 已初始化")
    except Exception as e:
        print(f"⚠️  GEE初始化失败: {e}")
        print("请运行: ee.Authenticate()")
except ImportError:
    print("⚠️  未安装 earthengine-api")
    print("安装命令: pip install earthengine-api")

# 导入智能更新模块
try:
    from smart_dataset_discoverer import SmartDatasetDiscoverer, DatasetVersionTracker
    from smart_update_manager import SmartUpdateManager
    print("✅ 智能更新模块导入成功")
except Exception as e:
    print(f"⚠️  模块导入失败: {e}")

print("\n" + "="*60)
print("✅ 环境初始化完成！")
print("="*60)

### 第2步：检查可用更新

In [ ]:
# ============================================================
# 检查数据集更新
# ============================================================

# 创建更新管理器
config_path = project_root / 'config' / 'datasets_config.json'
manager = SmartUpdateManager(str(config_path))

# 检查更新
print("🔍 正在检查数据集更新...\n")
updates = manager.check_updates()

if updates:
    print(f"\n✅ 发现 {len(updates)} 个可用更新")
    
    # 显示更新详情
    for data_type, info in updates.items():
        print(f"\n📊 {data_type}:")
        print(f"   当前版本: {info['current_version']}")
        print(f"   推荐版本: {info['recommended_version']}")
        print(f"   原因: {info['reason']}")
else:
    print("\n✅ 所有数据源已是最新版本，无需更新")

### 第3步：应用更新（可选）

In [ ]:
# ============================================================
# 应用更新
# ============================================================

# 注意：运行此单元将更新配置文件
# 如果不想自动更新，将 auto_apply 设置为 False

auto_apply = False  # 设置为 True 自动应用更新

if auto_apply and updates:
    print("\n🔄 正在应用更新...")
    result = manager.update_all(auto_apply=True, create_backup=True)
    
    if result['success']:
        print(f"\n✅ 更新成功！共更新 {result['updated_count']} 个数据源")
        print(f"💾 备份文件: {result.get('backup_file', 'N/A')}")
    else:
        print(f"\n❌ 更新失败: {result['message']}")
else:
    print("\nℹ️  跳过自动更新")
    print("💡 提示：将 auto_apply 设置为 True 以应用更新")

---

## 🔧 高级功能

### 1. 查看数据集详细信息

In [ ]:
# ============================================================
# 获取数据集详细信息
# ============================================================

# 创建发现器
discoverer = SmartDatasetDiscoverer()

# 选择要查看的数据类型
data_type = "LST"  # 可以改为: NDVI, EVI, Albedo, PM25

print(f"\n📊 获取 {data_type} 数据集详细信息...\n")

# 获取推荐
recommendation = discoverer.recommend_dataset(data_type)

if 'error' not in recommendation:
    print(f"推荐集合: {recommendation['recommended_collection']}")
    print(f"版本: {recommendation['version']}")
    print(f"\n可用波段:")
    for band in recommendation['bands']:
        print(f"  - {band}")
    
    if recommendation.get('time_range'):
        print(f"\n时间范围:")
        print(f"  开始: {recommendation['time_range']['start']}")
        print(f"  结束: {recommendation['time_range']['end']}")
else:
    print(f"错误: {recommendation['error']}")

### 2. 比较两个数据集

In [ ]:
# ============================================================
# 比较数据集
# ============================================================

# 比较不同版本的MODIS LST数据集
dataset1 = "MODIS/061/MOD11A2"  # 最新版本
dataset2 = "MODIS/006/MOD11A2"  # 旧版本

print(f"\n🔍 比较 {dataset1} 和 {dataset2}...\n")

comparison = discoverer.compare_datasets(dataset1, dataset2)

print("数据集1:")
print(f"  集合: {dataset1}")
print(f"  波段数: {len(comparison['dataset1'].get('bands', []))}")

print("\n数据集2:")
print(f"  集合: {dataset2}")
print(f"  波段数: {len(comparison['dataset2'].get('bands', []))}")

print("\n差异:")
print(f"  更新版本: {comparison['differences']['version_newer']}")
print(f"  更多波段: {comparison['differences']['more_bands']}")

### 3. 回滚到之前的配置

In [ ]:
# ============================================================
# 回滚配置
# ============================================================

# 注意：只有在之前更新过的情况下才能回滚

rollback = False  # 设置为 True 以执行回滚

if rollback:
    print("\n⏪ 正在回滚到之前的配置...\n")
    
    success = manager.rollback_update()
    
    if success:
        print("✅ 回滚成功！")
    else:
        print("❌ 回滚失败")
else:
    print("\nℹ️  跳过回滚")
    print("💡 提示：将 rollback 设置为 True 以执行回滚")

### 4. 导出更新报告

In [ ]:
# ============================================================
# 导出更新报告
# ============================================================

output_path = project_root / 'output' / f'update_report_{datetime.now().strftime("%Y%m%d_%H%M%S")}.json'

print("\n📄 正在生成更新报告...\n")

manager.export_update_report(str(output_path))

print(f"\n✅ 报告已保存到: {output_path}")

---

## 📊 完整更新流程

下面的代码展示了完整的更新流程，包括所有步骤：

In [ ]:
# ============================================================
# 完整更新流程
# ============================================================

def full_update_workflow():
    """完整的更新工作流程"""
    
    print("\n" + "="*60)
    print("🚀 智能数据更新系统 - 完整流程")
    print("="*60)
    
    # 步骤1：初始化
    print("\n步骤1: 初始化")
    print("-" * 60)
    manager = SmartUpdateManager()
    print("✅ 管理器已创建")
    
    # 步骤2：检查更新
    print("\n步骤2: 检查更新")
    print("-" * 60)
    updates = manager.check_updates()
    
    if not updates:
        print("\n✅ 所有数据源已是最新版本")
        return {'success': True, 'message': '已是最新版本'}
    
    # 步骤3：显示更新摘要
    print("\n步骤3: 更新摘要")
    print("-" * 60)
    for data_type, info in updates.items():
        print(f"{data_type}: {info['current_version']} → {info['recommended_version']}")
    
    # 步骤4：应用更新
    print("\n步骤4: 应用更新")
    print("-" * 60)
    result = manager.update_all(auto_apply=True, create_backup=True)
    
    if not result['success']:
        print(f"❌ 更新失败: {result['message']}")
        return result
    
    # 步骤5：验证更新
    print("\n步骤5: 验证更新")
    print("-" * 60)
    validation = result.get('validation', {})
    
    if validation.get('valid'):
        print("✅ 所有数据源验证通过")
    else:
        print("⚠️  验证失败：")
        for error in validation.get('errors', []):
            print(f"  - {error}")
    
    if validation.get('warnings'):
        print("\n⚠️  警告：")
        for warning in validation.get('warnings', []):
            print(f"  - {warning}")
    
    # 步骤6：导出报告
    print("\n步骤6: 导出报告")
    print("-" * 60)
    output_path = project_root / 'output' / f'update_report_{datetime.now().strftime("%Y%m%d_%H%M%S")}.json'
    manager.export_update_report(str(output_path))
    print(f"✅ 报告已保存: {output_path}")
    
    print("\n" + "="*60)
    print("✅ 更新流程完成！")
    print("="*60)
    
    return result

# 运行完整流程
# 取消下面的注释以运行
# result = full_update_workflow()

---

## 💡 使用技巧

### 定期检查更新

建议每月或每季度运行一次更新检查：

```python
# 每月检查一次
manager = SmartUpdateManager()
updates = manager.check_updates()
```

### 安全更新

1. **总是创建备份** - 默认开启
2. **先测试再应用** - 使用 `auto_apply=False`
3. **验证更新** - 自动验证所有数据源
4. **保存报告** - 便于追踪和审计

### 自定义配置

你可以自定义配置文件路径：

```python
# 使用自定义配置
manager = SmartUpdateManager('path/to/your/config.json')
```

---

## 📞 获取帮助

### 常见问题

**Q: GEE初始化失败怎么办？**

```python
import ee
ee.Authenticate()  # 认证
ee.Initialize()    # 初始化
```

**Q: 如何回滚到之前的配置？**

```python
manager.rollback_update()
```

**Q: 更新后数据提取失败？**

1. 检查验证结果中的错误信息
2. 尝试回滚到之前的配置
3. 查看更新报告了解详细信息

---

*智能数据更新系统 - v1.0*  
*最后更新: 2026-03-13*